# TENSORFLOW IMPLEMENTATION

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models

In [2]:
# Parameters
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Loading Datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    'dermal_vision_dataset/train',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    'dermal_vision_dataset/val',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

Found 15427 files belonging to 25 classes.
Found 3311 files belonging to 25 classes.


In [3]:
# Optimization for performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [4]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.1),
    layers.RandomBrightness(0.1),
])

In [5]:
base_model = tf.keras.applications.EfficientNetV2B0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze the base model
base_model.trainable = False

24274472/24274472 ━━━━━━━━━━━━━━━━━━━━ 7s 0us/step


In [6]:
# Build the final model
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.3),  # Prevents overfitting
    layers.Dense(25, activation='softmax') # Your 25 categories
])

In [7]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [9]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', 
        patience=3, 
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath='core2cover_v1.keras',
        save_best_only=True
    )
]

In [10]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

Epoch 1/20
483/483 ━━━━━━━━━━━━━━━━━━━━ 1352s 3s/step - accuracy: 0.2997 - loss: 2.6904 - val_accuracy: 0.3872 - val_loss: 2.1320
Epoch 2/20
483/483 ━━━━━━━━━━━━━━━━━━━━ 1197s 2s/step - accuracy: 0.3622 - loss: 2.2822 - val_accuracy: 0.3981 - val_loss: 2.0659
Epoch 3/20
483/483 ━━━━━━━━━━━━━━━━━━━━ 1356s 3s/step - accuracy: 0.3786 - loss: 2.1742 - val_accuracy: 0.3987 - val_loss: 2.0360
Epoch 4/20
483/483 ━━━━━━━━━━━━━━━━━━━━ 1265s 3s/step - accuracy: 0.3924 - loss: 2.1118 - val_accuracy: 0.4177 - val_loss: 2.0062
Epoch 5/20
483/483 ━━━━━━━━━━━━━━━━━━━━ 247s 509ms/step - accuracy: 0.4009 - loss: 2.0739 - val_accuracy: 0.4162 - val_loss: 1.9767
Epoch 6/20
267/483 ━━━━━━━━━━━━━━━━━━━━ 1:21 379ms/step - accuracy: 0.4105 - loss: 2.0234

KeyboardInterrupt: 

# RESTART WITH PYTORCH

In [1]:
import torch

print(f"Is PyTorch using the RTX 3050? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

Is PyTorch using the RTX 3050? True
Device Name: NVIDIA GeForce RTX 3050 Laptop GPU


### 1. Data Loading & Augmentation

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os

In [3]:
# Set Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
device

device(type='cuda')

In [5]:
# 1. Data Augmentation
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(0.2 * 180), # Convert to degrees
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

In [6]:
# 2. Loading Datasets
image_datasets = {
    x: datasets.ImageFolder(os.path.join('dermal_vision_dataset', x), data_transforms[x])
    for x in ['train', 'val']
}

dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
    for x in ['train', 'val']
}

In [7]:
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

### 2. Model Architecture (EfficientNet-V2B0)

In [8]:
# 1. Load Pre-trained EfficientNet-V2-S
model = models.efficientnet_v2_s(weights='DEFAULT')

In [9]:
# 2. Freeze the Base Model
for param in model.parameters():
    param.requires_grad = False

In [10]:
# 3. Build the Final Model Head
# Replace the classifier with your specific architecture
num_ftrs = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.BatchNorm1d(num_ftrs), # Replicating BatchNormalization
    nn.Linear(num_ftrs, 25)    # 25 Categories
)

model = model.to(device)

In [11]:
# 4. Define Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

### 3. Training the Model

In [12]:
import time
import copy

In [13]:
def train_model(model, criterion, optimizer, num_epochs=20, patience=3):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss = float('inf')
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch + 1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # ModelCheckpoint & EarlyStopping logic
            if phase == 'val':
                if epoch_loss < best_loss:
                    best_loss = epoch_loss
                    best_model_wts = copy.deepcopy(model.state_dict())
                    torch.save(model.state_dict(), 'core2cover_v1.pth')
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1
        
        if epochs_no_improve >= patience:
            print("Early Stopping triggered!")
            break

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    model.load_state_dict(best_model_wts)
    return model

In [15]:
# START TRAINING
trained_model = train_model(model, criterion, optimizer, num_epochs=20)

Epoch 1/20
----------
train Loss: 2.5129 Acc: 0.2885
val Loss: 2.3238 Acc: 0.3334
Epoch 2/20
----------
train Loss: 2.3309 Acc: 0.3333
val Loss: 2.2518 Acc: 0.3561
Epoch 3/20
----------
train Loss: 2.3117 Acc: 0.3399
val Loss: 2.2289 Acc: 0.3576
Epoch 4/20
----------
train Loss: 2.2897 Acc: 0.3413
val Loss: 2.1976 Acc: 0.3685
Epoch 5/20
----------
train Loss: 2.2826 Acc: 0.3459
val Loss: 2.2212 Acc: 0.3600
Epoch 6/20
----------
train Loss: 2.2628 Acc: 0.3504
val Loss: 2.1925 Acc: 0.3745
Epoch 7/20
----------
train Loss: 2.2656 Acc: 0.3455
val Loss: 2.2019 Acc: 0.3618
Epoch 8/20
----------
train Loss: 2.2432 Acc: 0.3491
val Loss: 2.1884 Acc: 0.3818
Epoch 9/20
----------
train Loss: 2.2392 Acc: 0.3499
val Loss: 2.1557 Acc: 0.3827
Epoch 10/20
----------
train Loss: 2.2448 Acc: 0.3502
val Loss: 2.1775 Acc: 0.3651
Epoch 11/20
----------
train Loss: 2.2331 Acc: 0.3573
val Loss: 2.1756 Acc: 0.3712
Epoch 12/20
----------
train Loss: 2.2274 Acc: 0.3551
val Loss: 2.1633 Acc: 0.3763
Early Stoppin

### CLEARING GPU MEMORY

In [14]:
import torch
import gc

# 1. Delete the heavy objects from your current session
# Replace 'model' and 'optimizer' with your actual variable names if they differ
if 'model' in globals():
    del model
if 'optimizer' in globals():
    del optimizer

# 2. Force Python to find and clear unreferenced objects
gc.collect()

# 3. The "Magic" Command: Flush the specialized PyTorch GPU cache
torch.cuda.empty_cache()

# 4. Optional: Verify the memory is actually back
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

Allocated: 0.00 MB
Reserved: 0.00 MB


### 4. Fine-Tuning the Model

In [15]:
# 1. Load Pre-trained EfficientNet-V2-S
model = models.efficientnet_v2_s(weights='DEFAULT')

In [16]:
# 1. First, freeze EVERYTHING to reset
for param in model.parameters():
    param.requires_grad = False

In [17]:
# 2. Unfreeze only the last two stages (Stage 6 and 7)
# This keeps the VRAM usage low while giving the model 'clinical' vision
for param in model.features[6:].parameters():
    param.requires_grad = True

In [18]:
# 3. Always unfreeze the classifier head
for param in model.classifier.parameters():
    param.requires_grad = True

In [19]:
# 4. Use a very low learning rate
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)
patience=5

In [20]:
# 5. Push back to GPU to be safe
model = model.to(device)

In [21]:
# Now run your training call
model_ft = train_model(
    model, 
    criterion, 
    optimizer, 
    num_epochs=15, 
    patience=5
)

Epoch 1/15
----------
train Loss: 5.6644 Acc: 0.0785
val Loss: 4.0830 Acc: 0.1253
Epoch 2/15
----------
train Loss: 3.2261 Acc: 0.2162
val Loss: 2.8429 Acc: 0.2652
Epoch 3/15
----------
train Loss: 2.6347 Acc: 0.2899
val Loss: 2.4474 Acc: 0.3277
Epoch 4/15
----------
train Loss: 2.4098 Acc: 0.3257
val Loss: 2.2951 Acc: 0.3612
Epoch 5/15
----------
train Loss: 2.2754 Acc: 0.3563
val Loss: 2.1676 Acc: 0.3839
Epoch 6/15
----------
train Loss: 2.1880 Acc: 0.3742
val Loss: 2.0819 Acc: 0.4011
Epoch 7/15
----------
train Loss: 2.1081 Acc: 0.3945
val Loss: 2.0357 Acc: 0.4114
Epoch 8/15
----------
train Loss: 2.0401 Acc: 0.4103
val Loss: 1.9719 Acc: 0.4298
Epoch 9/15
----------
train Loss: 1.9753 Acc: 0.4233
val Loss: 1.9197 Acc: 0.4382
Epoch 10/15
----------
train Loss: 1.9134 Acc: 0.4473
val Loss: 1.8967 Acc: 0.4434
Epoch 11/15
----------
train Loss: 1.8673 Acc: 0.4545
val Loss: 1.8674 Acc: 0.4491
Epoch 12/15
----------
train Loss: 1.8261 Acc: 0.4666
val Loss: 1.8113 Acc: 0.4615
Epoch 13/15
-

### 5. Advanced Hyperparameter Optimization

In [21]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from torch.optim.lr_scheduler import ReduceLROnPlateau
import copy
import time

In [22]:
# 1. Calculate Class Weights mathematically
# We extract the true labels from your dataset to analyze the imbalance
train_targets = image_datasets['train'].targets
unique_classes = np.unique(train_targets)

In [23]:
# compute_class_weight automatically assigns higher values to rare diseases
weights = compute_class_weight(class_weight='balanced', classes=unique_classes, y=train_targets)
class_weights_tensor = torch.tensor(weights, dtype=torch.float).to(device)
print("✅ Class Weights calculated and moved to GPU.")

✅ Class Weights calculated and moved to GPU.


In [24]:
# 2. Update the Criterion (Loss Function) to use these weights
criterion_weighted = nn.CrossEntropyLoss(weight=class_weights_tensor)

# 3. Setup Optimizer and the new Dynamic Scheduler
# We keep the layers unfrozen exactly as they were in the previous step
optimizer_adv = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)

# If val_loss doesn't drop for 2 consecutive epochs, cut learning rate by half
scheduler = ReduceLROnPlateau(optimizer_adv, mode='min', factor=0.5, patience=2)

In [25]:
# 4. Advanced Training Loop (incorporating the Scheduler)
def train_model_advanced(model, criterion, optimizer, scheduler, num_epochs=15, patience=5):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss = float('inf')
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch + 1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Scheduler & Early Stopping Logic
            if phase == 'val':
                # Step the scheduler based on validation loss
                scheduler.step(epoch_loss)
                
                # Print current learning rate so you can watch it drop
                current_lr = optimizer.param_groups[0]['lr']
                print(f'Current Learning Rate: {current_lr}')

                if epoch_loss < best_loss:
                    best_loss = epoch_loss
                    best_model_wts = copy.deepcopy(model.state_dict())
                    
                    # Saving as a NEW file to preserve previous progress
                    torch.save(model.state_dict(), 'skin_model_adv.pth')
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1
        
        if epochs_no_improve >= patience:
            print(f"🛑 Early Stopping triggered after {epoch + 1} epochs!")
            break

    time_elapsed = time.time() - since
    print(f'Advanced Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    model.load_state_dict(best_model_wts)
    return model

In [26]:
# 5. START ADVANCED TRAINING
model_adv = train_model_advanced(model, criterion_weighted, optimizer_adv, scheduler, num_epochs=20)

Epoch 1/20
----------


RuntimeError: weight tensor should be defined either for all 1000 classes or no classes but got weight tensor of shape: [25]